# Course 2 : Celestial mechanics (three body problem)

In [ ]:
import numpy as np
from scipy.integrate import solve_ivp
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio
pio.templates.default = "seaborn"

Let us consider the Sun-Jupiter-Saturn system, where for simplicity we neglect the other bodies and influences in the solar system. In 1687, Isaac Newton, inspired by the three laws of Kepler, proposes the universal law of gravitation, that all cosmic objects attract each other pairwise with equal forces (but in opposite directions) proportional to the product of their masses and inversely proportional to the square of the distance between them.

The gravitational force $\vec F_{S\rightarrow P}$  applied by a body $S$ to a body $P$ is given by the following formula:

$$
\vec F_{S\rightarrow P} = - \vec F_{P\rightarrow S} = - \frac{G\,m_S\,m_P}{d^2} \vec{u}, 
$$

where G is the universal constant of gravitation, $m_S$ , $m_P$ are the masses of the bodies $S$ and $P$ , $d$ is the (Euclidean) distance between $S$ and $P$, and $\vec{u}$ is a vector with unit length in the direction from $S$ to $P$.

We represent the positions of the Sun, Jupiter and Saturn by three functions of time, $q_i(t) \in \mathbb{R}^3, i \in 0,1,2$ where the index $i = 0$ corresponds to the Sun, $i = 1$ corresponds to Jupiter, and $i = 2$ corresponds to Saturn. The respective masses of the three bodies are denoted by $m_i, i \in 0, 1, 2$. We also consider the momenta $p_i(t) = m_i q_i^′(t) \in \mathbb{R}^3, i \in 0, 1, 2$. Newton’s second law of dynamics then reads

\begin{equation}
{\mathrm d}_t p_0 = \vec F_{Sa\rightarrow S} + \vec F_{J\rightarrow S},\quad
{\mathrm d}_t p_1 = \vec F_{S\rightarrow J} + \vec F_{Sa\rightarrow J} ,\quad
{\mathrm d}_t p_2 = \vec F_{S\rightarrow Sa} + \vec F_{J\rightarrow Sa}.
\end{equation}

We thus end up with a system of 6 equations (or more precisely 18, since each $p_i$ and $q_i$ belongs to $\mathbb{R}^3$):

$$
\left\{\begin{aligned}
q_i^′(t) & = \frac{1}{m_i} p_i\\
p_i^′(t) & = -G \sum_{j \neq i} \frac{m_i m_j}{\lVert q_i - q_j \rVert^2} \frac{q_i - q_j}{\lVert q_i - q_j \rVert}
\end{aligned}\right.
\quad \quad i \in 0, 1, 2.
$$

In [ ]:
class three_body_model:

    def __init__(self):
        self.m0 = 1.00000597682
        self.m1 = 9.54786104043e-4
        self.m2 = 2.85583733151e-4
        self.G = 2.95912208286e-4

    def fcn(self, t, y):
        q = y[0:9]
        p = y[9:18]
        
        q_dot = np.zeros(9)
        p_dot = np.zeros(9)
        
        m0 = self.m0
        m1 = self.m1
        m2 = self.m2
        G  = self.G
        
        q_dot[0:3] = p[0:3]/m0
        q_dot[3:6] = p[3:6]/m1
        q_dot[6:9] = p[6:9]/m2
        
        p_dot[0:3] = -( G*m0*m1*((q[0:3]-q[3:6])/np.power(np.linalg.norm(q[0:3]-q[3:6]),3)) 
                       +G*m0*m2*((q[0:3]-q[6:9])/np.power(np.linalg.norm(q[0:3]-q[6:9]),3)))
        p_dot[3:6] = -( G*m1*m0*((q[3:6]-q[0:3])/np.power(np.linalg.norm(q[3:6]-q[0:3]),3))
                       +G*m1*m2*((q[3:6]-q[6:9])/np.power(np.linalg.norm(q[3:6]-q[6:9]),3)))
        p_dot[6:9] = -( G*m2*m0*((q[6:9]-q[0:3])/np.power(np.linalg.norm(q[6:9]-q[0:3]),3)) 
                       +G*m2*m1*((q[6:9]-q[3:6])/np.power(np.linalg.norm(q[6:9]-q[3:6]),3)))
        
        return np.concatenate((q_dot, p_dot))
    
    def hamiltonian(self, y):
        m0 = self.m0
        m1 = self.m1
        m2 = self.m2
        G = self.G
        nt = y.shape[1]
        neq = y.shape[0]
        q = y[0:neq//2]
        p = y[neq//2:neq]
        ham = np.zeros(nt)

        for i in range(nt):
            ham[i] = ( 0.5*( (1/m0)*np.dot(p[0:3,i],p[0:3,i]) 
                            +(1/m1)*np.dot(p[3:6,i],p[3:6,i]) 
                            +(1/m2)*np.dot(p[6:9,i],p[6:9,i]) ) 
                      - G *( (m0*m1)/np.linalg.norm(q[0:3,i]-q[3:6,i]) 
                            +(m0*m2)/np.linalg.norm(q[0:3,i]-q[6:9,i])
                            +(m1*m2)/np.linalg.norm(q[3:6,i]-q[6:9,i]) ) )

        return ham

## Numerical solution

In [ ]:
# time expressed in days
tini = 0.
tend = 15000.

In [ ]:
# initialization
m1 = 9.54786104043e-4
m2 = 2.85583733151e-4

qini = np.zeros(9)
qini[0] =  0.;        qini[1] =  0.;        qini[2] =  0.
qini[3] = -3.5023653; qini[4] = -3.8169847; qini[5] = -1.5507963
qini[6] =  9.0755314; qini[7] = -3.0458353; qini[8] = -1.6483708

pini = np.zeros(9)
pini[0] =  0.;            pini[1] =  0.;            pini[2] =  0.
pini[3] =  0.00565429*m1; pini[4] = -0.00412490*m1; pini[5] = -0.00190589*m1
pini[6] =  0.00168318*m2; pini[7] =  0.00483525*m2; pini[8] =  0.00192462*m2

yini = np.concatenate((qini, pini))

tbm = three_body_model()
fcn = tbm.fcn

sol = solve_ivp(fcn, (tini, tend), yini, method="RK45", rtol=1.e-12, atol=1.e-12)
fig = go.Figure()
fig.add_trace(go.Scatter(x=[0.], y=[0.], mode="markers", marker=dict(size=12, color="orange"), name="Sun"))
fig.add_trace(go.Scatter(x=[qini[3]], y=[qini[4]], mode="markers", marker=dict(size=12, color="olive"), showlegend=False))
fig.add_trace(go.Scatter(x=sol.y[3]-sol.y[0], y=sol.y[4]-sol.y[1], marker_color="olive", name="Jupiter"))
fig.add_trace(go.Scatter(x=[qini[6]], y=[qini[7]], mode="markers", marker=dict(size=12, color="steelblue"), showlegend=False))
fig.add_trace(go.Scatter(x=sol.y[6]-sol.y[0], y=sol.y[7]-sol.y[1], marker_color="steelblue ", name="Saturn"))
fig.update_layout(height=500, xaxis_range=[-12, 12])
fig.show()

## Hamiltonian

The quantity : 

$$
H = \frac{1}{2}\left(\frac{\lVert p_0 \rVert^2}{m_0} + \frac{\lVert p_1 \rVert^2}{m_1} + \frac{\lVert p_2 \rVert^2}{m_2} \right)               - G \left( \frac{m_0 m_1}{\lVert q_0 - q_1 \rVert} + \frac{m_0 m_2}{\lVert q_0 - q_2 \rVert} + \frac{m_1 m_2}{\lVert q_1 - q_2 \rVert} \right)
$$

is conserved.

In [ ]:
ham = tbm.hamiltonian(sol.y)

fig = go.Figure()
fig.add_trace(go.Scatter(x=sol.t, y=ham))
fig.update_layout(title="Hamiltonian")
fig.update_yaxes(range=[-3.15636e-8, -3.15634e-8], exponentformat='e')
fig.show()